# 🎓 MSEB: End-to-End Transcription Pipeline

Welcome to this hands-on lab for the **Massive Speech Evaluation Benchmark (MSEB)**. 

In this notebook, we will evaluate the **Whisper Base** model's transcription accuracy using the English (US) subset of the **Simple Voice Questions (SVQ)** dataset. 

### 💡 The Core Engineering Challenge
MSEB is an incredibly powerful, enterprise-grade evaluation framework. However, it was originally architected to run on internal distributed infrastructure using massive 22GB binary data archives (Parquet/Tar). 

**The Goal of this Lab:** We are going to adapt this enterprise pipeline to run smoothly in a standard, public cloud notebook environment. To do this, we will write several "Monkey Patches" (runtime code overrides) to bypass internal network requests, strict data schemas, and dependency collisions. 

By the end of this notebook, you will not only know how to run MSEB, but you will understand the deep Python mechanics of *why* and *how* we bend the rules to make it work!

## 🛠️ Phase 1: Environment & Dependency Resolution
When installing massive ML ecosystems, dependency collisions are common. In this setup phase, we must carefully orchestrate our installations to ensure Python 3.12 compatibility.

### Why are we doing this?
1. **The `rapidfuzz` Crash:** MSEB's grader relies on a math library called `jiwer`, which hardcodes a requirement for `rapidfuzz 2.13.7`. However, that older version cannot compile its C++ backend on modern Python 3.12 environments, causing the grader to fall back to a broken Python script and crash. We will forcefully override this rule and install the modern `3.x` C++ binaries.
2. **The `scann` Bypass:** ScaNN is a heavy vector-retrieval library used for text tasks. Compiling it takes 10+ minutes. Since we are only doing audio transcription, we will use Python's `MagicMock` to create a "phantom" module, tricking MSEB into thinking it's installed so we can skip the compile time.

In [ ]:
# 1. Purge conflicting packages to start with a clean slate
!pip uninstall -y whisper rapidfuzz jiwer -q

# 2. Install the core MSEB ecosystem FIRST (Explicitly including apache-beam)
# Note: This will secretly install the broken rapidfuzz 2.13.7 because 'jiwer' demands it.
!pip install git+https://github.com/google-research/mseb.git openai-whisper datasets soundfile crcmod apache-beam "pillow<11.0" jiwer -q

# 3. THE C++ OVERRIDE 
# We explicitly force the modern 3.x C++ binaries. 
# (Note: pip will print a red dependency warning here. We WANT this warning! It means our override worked.)
!pip install "rapidfuzz>=3.4.0" --upgrade --force-reinstall -q

import sys
from unittest.mock import MagicMock

# 4. SCANN BYPASS (Save 10 minutes of compile time)
mock_scann = MagicMock()
sys.modules['scann'] = mock_scann
sys.modules['scann.scann_ops'] = mock_scann
sys.modules['scann.scann_ops.py'] = mock_scann
sys.modules['scann.scann_ops.py.scann_ops_pybind'] = mock_scann

print("✅ Phase 1 Complete: Environment Mocked and C++ Math Engine Enforced.")

## 💾 Phase 2: Dynamic Dataset Localization
The official SVQ dataset is ~22GB. Downloading that into a temporary notebook instance is a waste of time and compute. 

### Why are we doing this?
Instead of downloading the entire archive, we will use Hugging Face's `streaming=True` feature to download just the first **20 audio files**, saving them as standard `.wav` files. 

Because we aren't using the official distributed `.parquet` archives, we must manually construct the two required MSEB metadata schemas:
* **`utt_index.jsonl` (The Master Directory):** Tells MSEB where the audio files live on the local hard drive.
* **`speech_transcription.jsonl` (The Task Manifest):** Provides the "Ground Truth" text that the grader uses to score the AI. 

*Note: Watch how we carefully inject keys like `"environment"` and `"transcript_truth"`. If these specific keys are missing, the MSEB grader will throw a `KeyError`!*

In [ ]:
import os
import json
import io
import soundfile as sf
from datasets import load_dataset

base_path = './mseb_data/svq'
audio_dir = os.path.join(base_path, 'audio')
os.makedirs(audio_dir, exist_ok=True)

index_path = os.path.join(base_path, 'utt_index.jsonl')
task_path = os.path.join(base_path, 'speech_transcription.jsonl')

print("🌐 Streaming real en_us dataset from Hugging Face...")
audio_ds = load_dataset("google/svq", "audio", split="test", streaming=True)

# Filter strictly for American English
en_stream = audio_ds.filter(lambda x: x['locale'].lower() == 'en_us')

num_examples = 20
print(f"⬇️ Downloading {num_examples} examples to local cache...")

with open(index_path, 'w') as f_idx, open(task_path, 'w') as f_task:
    for i, sample in enumerate(en_stream):
        if i >= num_examples: break
            
        # Extract audio bytes and save as a standard .wav file
        utt_id = sample['utt_id']
        audio_bytes = sample['waveform']['bytes']
        audio_arr, sr = sf.read(io.BytesIO(audio_bytes))
        rel_audio_path = f"audio/{utt_id}.wav"
        sf.write(os.path.join(base_path, rel_audio_path), audio_arr, sr)

        # 1. Master Metadata (Notice 'index' is a string path, not an integer)
        f_idx.write(json.dumps({
            "utt_id": utt_id, "locale": "en_us", 
            "speaker_id": sample.get('speaker_id', "speaker_000"),
            "speaker_age": sample.get('speaker_age', 30),
            "speaker_gender": sample.get('speaker_gender', "unknown"),
            "environment": sample.get('environment', "clean"),
            "text": sample.get('text', ''), "index": rel_audio_path 
        }) + '\n')

        # 2. Task Metadata (Contains the 'transcript_truth' for the math grader)
        f_task.write(json.dumps({
            "utt_id": utt_id, "text": sample.get('text', ''),
            "transcript": sample.get('text', ''), 
            "transcript_truth": sample.get('text', ''), 
            "locale": "en_us", "environment": sample.get('environment', "clean") 
        }) + '\n')

print("✅ Phase 2 Complete: 20-file mini-dataset localized with strict MSEB schema!")

## 🚀 Phase 3: Pipeline Execution & Final Patches
This is where we execute the core logic: `Encoder → Runner → Task → Leaderboard`. 

Because we are running this outside of the original internal network, we must deploy three final runtime patches to keep the pipeline stable:

### Why are we doing this?
1.  **The Apache Beam Patch:** Modern versions of Apache Beam hide their submodules to save memory. When MSEB tries to read Beam's type hints, it crashes. We manually build a "Frankenstein" namespace to feed the exact submodules MSEB is looking for.
2.  **The Dynamic Inheritance Patch:** The MSEB grader calculates separate scores for noisy audio (e.g., "media_noise"). Because our 20-file dataset only contains "clean" audio, the grader divides by zero and crashes. We dynamically subclass the task to force it to skip the sub-task grading.
3.  **The Audio Loader Patch:** MSEB expects data inside massive binary archives. We overwrite the `sounds()` method to instruct the task to simply read our local `.wav` files.

In [ ]:
import os
import sys
import json
import soundfile as sf
from absl import flags
import tensorflow as tf

# 🔥 THE DEFINITIVE APACHE BEAM FIX (LEVEL 2) 🔥
import apache_beam as beam
from apache_beam.utils import windowed_value
from apache_beam.runners import PipelineRunner

# Construct BOTH missing namespaces and staple them to the root beam object
beam.utils = type('MockUtils', (object,), {'windowed_value': windowed_value})
beam.runners = type('MockRunners', (object,), {'PipelineRunner': PipelineRunner})

from mseb import leaderboard
from mseb import runner as runner_lib
from mseb import tasks
from mseb import types
from mseb.encoders import encoder_registry

# Force-load the SVQ Speech module into the registry
from mseb.tasks.transcriptions.speech.svq import *

# 1. Initialize ABSL flags (Bypass Colab's internal '-f' flag)
FLAGS = flags.FLAGS
if not FLAGS.is_parsed():
    FLAGS([''])

# 2. Configuration Parameters
encoder_name = 'whisper_base_speech_to_text'
task_name = 'SVQEnUsSpeechTranscription'
batch_size = 16 # Lowered for Whisper memory safety in Colab
num_threads = 1

# 3. Paths
# 👉 If you have the real dataset, change 'dataset_basepath' to your local folder!
FLAGS.dataset_basepath = './mseb_data/svq/'
FLAGS.task_cache_basepath = f'./cache/task/transcription/{task_name}'
runner_output_path = f'./cache/runner/{task_name}'

os.makedirs(FLAGS.task_cache_basepath, exist_ok=True)
os.makedirs(runner_output_path, exist_ok=True)

# 4. Load the Pipeline
print(f"\nLoading Encoder: {encoder_name}...")
encoder = encoder_registry.get_encoder_metadata(encoder_name).load()

print("Initializing DirectRunner...")
runner = runner_lib.DirectRunner(
    encoder=encoder,
    batch_size=batch_size,
    num_threads=num_threads,
    output_path=runner_output_path,
)

print(f"Loading Task: {task_name}...")
task_cls = tasks.get_task_by_name(task_name)

# 🔧 COLAB SHORTCUT: The Inheritance Patch
# Since sub_tasks is a read-only property, we subclass the task on the fly
# to force it to return only the master task, preventing the ZeroDivisionError.
class ColabSVQTask(task_cls):
    @property
    def sub_tasks(self):
        return ['speech_transcription']

# Initialize our custom subclass instead of the original!
active_task = ColabSVQTask()

# ======================================================================
# 🔧 COLAB SHORTCUT: THE MONKEY PATCH
# Because we saved standalone .wav files instead of the massive
# binary archives MSEB expects, we must override the audio loader.
#
# ⚠️ REMOVE THIS FUNCTION ENTIRELY IF USING THE OFFICIAL DATASET! ⚠️
# ======================================================================
def custom_sound_loader():
    task_file = os.path.join(FLAGS.dataset_basepath, 'speech_transcription.jsonl')
    with open(task_file, 'r') as f:
        for line in f:
            data = json.loads(line)
            utt_id = data['utt_id']
            transcript = data['transcript_truth']

            wav_path = os.path.join(FLAGS.dataset_basepath, f'audio/{utt_id}.wav')
            audio_arr, sr = sf.read(wav_path)

            context = types.SoundContextParams(
                id=utt_id,
                sample_rate=sr,
                length=len(audio_arr),
                text=transcript
            )
            yield types.Sound(waveform=audio_arr, context=context)

# Inject the patch into the task
active_task.sounds = custom_sound_loader
# ======================================================================

# 5. Run the Benchmark
print("\n🚀 Starting Benchmark Evaluation. Whisper is transcribing...")
active_task.setup(runner=runner)
results = leaderboard.run_benchmark(
    encoder_name=encoder_name,
    runner=runner,
    task=active_task
)

# 6. Output Results
print("\n" + "="*50)
print("📊 LEADERBOARD METRICS (WER / CER)")
print("="*50)
output_file = f'./{task_name}_{encoder_name}_results.jsonl'
with tf.io.gfile.GFile(output_file, 'w') as f:
    for result in results:
        json_str = result.to_json() if hasattr(result, 'to_json') else str(result)
        print(json_str)
        f.write(json_str + '\n')

print(f"\n✅ Results successfully saved to {output_file}")

## 📊 Understanding Your Results
When the pipeline finishes, it outputs a JSON report containing your model's performance on the 20 files. 

### Key Metrics to Watch:
* **`WER` (Word Error Rate):** The percentage of *words* the AI got wrong (e.g., `0.639` = 63.9% error rate). Lower is better. 0.0 is perfect accuracy.
* **`SER` (Sentence Error Rate):** The percentage of *audio files* that contained at least one mistake (e.g., `0.8` = 80%). This means 4 out of 20 files were transcribed flawlessly.
* **`NoResultRate`:** How often the model timed out or failed to return any text. 

*💡 **Pro Tip:** Try changing `encoder_name = 'whisper_base_speech_to_text'` to `whisper_large_v3_speech_to_text` in the code above and run it again. You will see the Word Error Rate drop significantly!*